# TP3 Methode d’Uzawa pour l’optimisation sous contrainte 
## 12/06/20266
#### Léo SENTES Mia PERROUIN

### Exercice 1:


#### 1. Ecriture des contraintes  

Nous avons deux contraintes de type égalité.

La première, $(x,1^n)=1$ et la deuxième $(x,e)=r_0$ on peut les écrire sous forme $Cx=d$

$$

* $\langle x, 1^m \rangle = \begin{pmatrix} x_1 \\ \vdots \\ x_m \end{pmatrix} \cdot \begin{pmatrix} 1 \\ \vdots \\ 1 \end{pmatrix} = 1 \implies \sum_{i=1}^m x_i = 1$
* $\langle x, e \rangle = r_0 \implies \begin{pmatrix} x_1 \\ \vdots \\ x_m \end{pmatrix} \cdot \begin{pmatrix} e_1 \\ \vdots \\ e_m \end{pmatrix} = r_0 \implies \sum_{i=1}^m x_i \cdot e_i = r_0$
$$C = \begin{pmatrix} 1 & \dots & 1 \\ e_1 & \dots & e_m \end{pmatrix} \quad \text{et} \quad d = \begin{pmatrix} 1 \\ r_0 \end{pmatrix}$$

On a ici $f(x)=0.5(Ax,x)$  et  $h(x)=Cx-d=0$ 

On peut écrire les conditions d'optimalité par:

$\begin{cases} \nabla f(x) + \sum_{i=1}^m \lambda_i \cdot \nabla h_i(x) = 0 \\ h(x) = 0 \end{cases}$
donc :
$\begin{cases} \nabla \left( \frac{1}{2} \langle Ax, x \rangle \right) + \sum_{i=1}^m \lambda_i \cdot \nabla(Cx - d)_i = 0 \\ (Cx - d) = 0 \end{cases}$
$\begin{cases}  Ax + \lambda^T \cdot C = 0 \\ Cx = d \end{cases}$

On peut traduire ça par un système linaire global:

$$\begin{pmatrix} 
A & C^T \\ 
C & 0 
\end{pmatrix} 
\begin{pmatrix} 
x \\ \lambda \end{pmatrix} 
= 
\begin{pmatrix} 
0 \\ 
d 
\end{pmatrix}
$$
du modèle $M \cdot Y = P$

#### 2.
#### a)
On choisit ici $n=5,   e [1,2,3,4,5] $ et $r_0=2.5$

In [24]:
import numpy as np
import math

N=5
e=np.array ([1 ,2 ,3 ,4 , 5])
r0=2.5
A=np.diag ( e/N )
R=np.random.rand (N , N )
A=A+0.1*np.dot( np.transpose ( R ) ,R )
C=np.array([[1,1,1,1,1],[1,2,3,4,5]])
Ct=np.array([[1,1],[1,2],[1,3],[1,4],[1,5]])
#résolution du problème M.Y=p
M=np.zeros((N+2,N+2))
for i in range (0,N+2):
    for j in range (0,N+2):
        if (i<N) and (j<N):
            M[i,j]=A[i,j]
        if ((i<N) and (j>=N)):
            M[i,j]=Ct[i,j-N]
        if ((i>=N) and (j<N)):
            M[i,j]=C[i-N,j]
p=([0,0,0,0,0,1,r0]) 
y=np.linalg.solve(M,p)
x=np.zeros(N)
lam=np.zeros(2)
for i in range (0,N+2):
    if i<N:
        x[i]=y[i]
    else:
        lam[i-N]=y[i]
print("La valeur du vecteur x est",x)

La valeur du vecteur x est [0.35523413 0.15710076 0.2112196  0.185322   0.09112351]


#### b)

In [33]:
def uzawa(rho,lam,eta,A,C,Ct,d):
    iter=0
    b=np.dot(Ct,lam)
    x=np.linalg.solve(A,-b)
    while (np.linalg.norm(np.dot(C,x)-d)>eta) and (iter<1000):
        lam=lam+rho*(np.dot(C,x)-d)
        b=np.dot(Ct,lam)
        x=np.linalg.solve(A,-b)
        iter=+1
    return x,lam


In [34]:
lam0=np.ones(2)
d=(1,2.5)
print("La valeur du vecteur x est",uzawa(0.01,lam0,0.0001,A,C,Ct,d))

La valeur du vecteur x est (array([0.35512518, 0.15707477, 0.21121774, 0.18533486, 0.09115143]), array([-0.18028768, -0.01567454]))


### Exercice 2: Optimisation avec contrainte


#### a)

In [58]:
def proj_convexe(C,d,y,rho,eps):
    k=0
    v_pi=np.ones(len(y))
    lam=np.ones(len(d))
    x0=y
    x1=y-np.dot(C.T,lam)
    while(np.linalg.norm(x1-x0)<eps):
        for i in range (0,len(y)):
            if (x1[i]<0):
                v_pi[i]=0
            else:
                v_pi[i]=x1[i]
        x0=x1
        k=+1
        lam=np.dot(v_pi,(lam+rho*(np.dot(C,x0)-d))) #problème de dimension ici
        x1=y-np.dot(C.T,lam)
    x0=x1            
    return x0, lam,k

#### b)

In [59]:
C=np.array([[1,1],[-1,-1],[1,-1],[-1,1]])
d=np.array([1,1,1,1])
y=np.array([-1,-2])
#on choisit y=(-2,2)
print(proj_convexe(C,d,y,0.01,0.0001))

ValueError: shapes (2,) and (4,) not aligned: 2 (dim 0) != 4 (dim 0)